# Agent Repair Pipeline — MuSiQue

**ICLR Experiment Pipeline — Colab A100-80GB + Qwen2.5-32B-Instruct-AWQ**

| Stage | What it does | ~Time (A100-80GB, 500 Qs) |
|---|---|---|
| 0 | Download MuSiQue, create folders | 1 min |
| 1 | ReAct trajectory generation (Qwen2.5-32B-Instruct-AWQ) | ~1-2 h |
| 2 | Step-level uncertainty scoring | ~1-2 h |
| 3 | 72B judge error annotation | ~1-2 h |
| 4 | Localization scoring (+ cascade-aware rules) | < 1 min |
| 5 | 126 repair strategies (backtrack × nudge type ablation) | ~4-6 h |
| 6 | Tables, stats, figures, ablation analysis | < 1 min |

**Every stage is resumable** — if Colab disconnects, re-run the cell and it picks up where it left off.

---
## 0. Setup

In [1]:
# Verify GPU — should show A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA A100-SXM4-80GB, 81920 MiB


In [2]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab_musique.yaml'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kishormorol/agent-repair.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

Cloning into '/content/agent-repair'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 239 (delta 125), reused 181 (delta 70), pack-reused 0 (from 0)
Receiving objects: 100% (239/239), 334.29 KiB | 8.57 MiB/s, done.
Resolving deltas: 100% (125/125), done.
Working directory: /content/agent-repair


In [4]:
# Check Colab's CUDA version first
!nvcc --version 2>/dev/null || echo "nvcc not found"
!python -c "import torch; print(f'torch CUDA: {torch.version.cuda}')" 2>/dev/null || echo "torch not installed yet"
!nvidia-smi | grep "CUDA Version"

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
torch CUDA: 12.8
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |


In [5]:
# Install vLLM — pinned to 0.19.0 (last version with CUDA 12 default wheels)
# vLLM >= 0.20 ships CUDA 13 wheels which don't work on Colab's CUDA 12.x
!pip uninstall -y vllm 2>/dev/null
!pip install -q "vllm==0.19.0" 2>&1 | tail -5

# Install remaining dependencies
!pip install -q "transformers>=4.57.0" "tokenizers>=0.21.0" accelerate \
    huggingface_hub datasets scipy scikit-learn pandas pyarrow \
    pyyaml tqdm matplotlib seaborn statsmodels 2>&1 | tail -3

print('\n--- Installation complete ---')

google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.

--- Installation complete ---


In [6]:
# Quick sanity check: can vLLM see the GPU?
import torch, vllm
print(f'torch:  {torch.__version__}')
print(f'vLLM:   {vllm.__version__}')
print(f'CUDA:   {torch.version.cuda}')
print(f'GPU:    {torch.cuda.get_device_name(0)}')
print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch:  2.10.0+cu128
vLLM:   0.19.0
CUDA:   12.8
GPU:    NVIDIA A100-SXM4-80GB
VRAM:   85.1 GB


In [7]:
# Clear previous data
import shutil, os
drive_base = '/content/drive/MyDrive/agent-repair-musique'
for d in ['outputs', 'data/processed']:
    p = os.path.join(drive_base, d)
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')
for d in ['outputs', 'data/processed']:
    p = os.path.join('/content/agent-repair', d)
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')
print('Ready for fresh run with MuSiQue')

Ready for fresh run with MuSiQue


In [23]:
!cd /content/agent-repair && git pull

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 815 bytes | 815.00 KiB/s, done.
From https://github.com/kishormorol/agent-repair
   bedc712..238a905  main       -> origin/main
Updating bedc712..238a905
Fast-forward
 src/env/musique_env.py | 12 +++++++++++-
 1 file changed, 11 insertions(+), 1 deletion(-)


---
## Smoke Test (20 questions, ~10-15 min)

**Run this first** to verify everything works before committing A100 units to the full 500-question run.

In [24]:
stages = ['run_setup', 'run_generate', 'run_uncertainty', 'run_annotate',
          'run_localize', 'run_repair', 'run_eval']

for stage in stages:
    print(f'\n{"="*60}\n  {stage}\n{"="*60}')
    !python scripts/{stage}.py --config {CONFIG} --limit 20
    print()


  run_setup
00:24:50 | INFO    | setup | config=config/config_colab_musique.yaml  base=/content/drive/MyDrive/agent-repair-musique
00:24:50 | INFO    | setup | Dataset: musique (file: musique_dev.json)
00:24:50 | INFO    | setup | Downloading musique via registry download function...
README.md: 100% 359/359 [00:00<00:00, 1.84MB/s]
musique_full_v1.0_train.jsonl: 100% 477M/477M [00:02<00:00, 220MB/s]
musique_full_v1.0_dev.jsonl: 100% 59.4M/59.4M [00:00<00:00, 62.1MB/s]
Generating train split: 100% 39876/39876 [00:01<00:00, 37260.61 examples/s]
Generating validation split: 100% 4834/4834 [00:00<00:00, 36527.01 examples/s]
  Loaded MuSiQue from bdsaglam/musique
00:25:02 | INFO    | setup | Downloaded 4834 records for musique.
00:25:03 | INFO    | setup | Dataset ready: /content/drive/MyDrive/agent-repair-musique/data/raw/musique_dev.json (61.1 MB, 4834 questions)
00:25:03 | INFO    | setup | Output dirs created under: /content/drive/MyDrive/agent-repair-musique


  run_generate
00:25:04 |

In [25]:
# Check smoke test produced output
import glob
tables = glob.glob('outputs/tables/*') + glob.glob('/content/drive/MyDrive/agent-repair-musique/outputs/tables/*')
figs = glob.glob('outputs/figures/*') + glob.glob('/content/drive/MyDrive/agent-repair-musique/outputs/figures/*')
print(f'Tables: {len(tables)}, Figures: {len(figs)}')
if tables or figs:
    print('Smoke test PASSED — safe to run the full pipeline below.')
else:
    print('WARNING: No outputs found. Check the logs above for errors.')

Tables: 4, Figures: 0
Smoke test PASSED — safe to run the full pipeline below.


---
## Full Pipeline (500 questions)

Only run these cells after the smoke test passes. To reset and run fresh, delete the `data/processed/` folder.

### Stage 0: Download MuSiQue

In [26]:
!python scripts/run_setup.py --config {CONFIG}

00:46:05 | INFO    | setup | config=config/config_colab_musique.yaml  base=/content/drive/MyDrive/agent-repair-musique
00:46:05 | INFO    | setup | Dataset: musique (file: musique_dev.json)
00:46:05 | INFO    | setup | Dataset already present: /content/drive/MyDrive/agent-repair-musique/data/raw/musique_dev.json
00:46:06 | INFO    | setup | Dataset ready: /content/drive/MyDrive/agent-repair-musique/data/raw/musique_dev.json (61.1 MB, 4834 questions)
00:46:06 | INFO    | setup | Output dirs created under: /content/drive/MyDrive/agent-repair-musique


### Stage 1: Generate ReAct Trajectories

In [27]:
!python scripts/run_generate.py --config {CONFIG}

00:46:07 | INFO    | stage1 | config=config/config_colab_musique.yaml  base=/content/drive/MyDrive/agent-repair-musique
00:46:07 | INFO    | stage1 | Dataset: musique (env: MuSiQueEnv)
00:46:07 | INFO    | stage1 | Pool: 500 questions
00:46:07 | INFO    | stage1 | To process: 479 of 500 (21 already done)
00:46:09 | INFO    | stage1 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-22 00:46:12.088414: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 00:46:12.161532: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild

### Stage 2: Step-Level Uncertainty

In [28]:
!python scripts/run_uncertainty.py --config {CONFIG}

00:54:13 | INFO    | stage2 | config=config/config_colab_musique.yaml  base=/content/drive/MyDrive/agent-repair-musique
00:54:13 | INFO    | stage2 | Math metrics: 455 of 475 trajectories to do
00:54:28 | INFO    | stage2 | Math metrics done.
00:54:28 | INFO    | stage2 | Sampling metrics: 409 of 425 failed trajectories
00:54:30 | INFO    | stage2 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-22 00:54:33.307067: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 00:54:33.380288: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other ope

### Stage 3: Error Annotation (72B Judge)

In [29]:
!python scripts/run_annotate.py --config {CONFIG}

01:12:07 | INFO    | stage3 | config=config/config_colab_musique.yaml  base=/content/drive/MyDrive/agent-repair-musique
01:12:07 | INFO    | stage3 | To annotate: 409 of 425 failed trajectories
01:12:08 | INFO    | stage3 | GPU VRAM: 85.094825984 GB | judge: Qwen/Qwen2.5-72B-Instruct-AWQ (85 GB -> 72B judge)
2026-07-22 01:12:12.080222: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 01:12:12.151543: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 07-22 01:12:22 [utils.py:233] non-default args: {'trust_rem

### Stage 4: Localization Scoring

In [30]:
!python scripts/run_localize.py --config {CONFIG}

01:23:06 | INFO    | stage4 | config=config/config_colab_musique.yaml  base=/content/drive/MyDrive/agent-repair-musique
01:23:08 | INFO    | stage4 | records: 4250 (425 trajectories x 10 metrics)
                        argmax_top1  argmax_within1    mrr  threshold_hit  threshold_within1  top2_hit  top3_hit    n
metric                                                                                                               
self_consistency              0.129           0.322  0.328          0.169              0.454     0.231     0.346  425
max_token_prob_max            0.099           0.282  0.296          0.141              0.440     0.200     0.282  425
token_entropy_max             0.099           0.242  0.292          0.155              0.369     0.200     0.287  425
verbalized_confidence         0.085           0.242  0.323          0.122              0.339     0.221     0.379  425
token_entropy_mean_k50        0.073           0.268  0.268          0.089              0.379    

### Stage 5: Repair (126 Strategies)

In [31]:
!python scripts/run_repair.py --config {CONFIG}

01:23:09 | INFO    | stage5 | config=config/config_colab_musique.yaml  base=/content/drive/MyDrive/agent-repair-musique
01:23:09 | INFO    | stage5 | Dataset: musique (env: MuSiQueEnv)
01:23:09 | INFO    | stage5 | 425 failed x 126 strategies x 3 seeds = 160650 result rows (actual GPU runs are far fewer: dedup)
01:23:10 | INFO    | stage5 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-22 01:23:14.095646: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 01:23:14.168204: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, 

### Stage 6: Evaluation

In [32]:
!python scripts/run_eval.py --config {CONFIG}

04:36:13 | INFO    | stage6 | config=config/config_colab_musique.yaml  base=/content/drive/MyDrive/agent-repair-musique
04:36:17 | INFO    | stage6 | rows: 161925 | uncertainty strategies: 120 (bt0: 60) | + ensemble

MAIN RESULTS  (fixed_% = share of FAILED trajectories that the repair fixed)
                                                           strategy  fixed_%     95%_CI  avg_tokens  avg_tool_calls  hit_true_step_%
                                                        random_step      3.6   2.6-4.6%         216            3.92             14.8
                                                       full_restart      6.4   5.0-7.7%         348            6.63             42.8
                                               oracle_targeted__bt2      6.1   4.9-7.4%         346            6.38             42.8
                                     oracle_targeted__bt2__informed      5.2   4.1-6.5%         347            6.45             42.8
                                         

---
## Results

In [33]:
from src.utils import load_config
cfg = load_config(CONFIG)

summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())
else:
    print('Summary not yet generated. Run Stage 6 first.')

Summary not yet generated. Run Stage 6 first.


In [34]:
import pandas as pd

results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)
else:
    print('Results table not yet generated. Run Stage 6 first.')

,strategy,fixed_%,95%_CI,avg_tokens,avg_tool_calls,hit_true_step_%
0,random_step,3.6,2.6-4.6%,216,3.92,14.8
1,full_restart,6.4,5.0-7.7%,348,6.63,42.8
2,oracle_targeted__bt2,6.1,4.9-7.4%,346,6.38,42.8
3,oracle_targeted__bt2__informed,5.2,4.1-6.5%,347,6.45,42.8
4,oracle_targeted__informed,4.4,3.3-5.6%,320,5.88,100.0
...,...,...,...,...,...,...
122,unc__verbalized_confidence__topk__bt2,4.5,3.4-5.7%,326,5.96,33.9
123,unc__verbalized_confidence__topk__bt2__informed,4.6,3.5-5.7%,323,6.03,33.9
124,unc__verbalized_confidence__topk__informed,3.8,2.8-4.9%,284,5.28,22.1
125,uncertainty_ensemble_any,16.1,14.1-18.1%,15314,281.52,84.9


In [35]:
import glob
from IPython.display import display, Image

fig_dir = cfg.path('figures')
figs = sorted(glob.glob(os.path.join(fig_dir, '*.png')))
if figs:
    for f in figs:
        print(f'\n--- {os.path.basename(f)} ---')
        display(Image(filename=f, width=800))
else:
    print('No figures yet. Run Stage 6 first.')

No figures yet. Run Stage 6 first.


---
## Re-run Stages 4–6 with Cascade-Aware Strategies

**Use this after you already have stages 1–3 completed** (trajectories, uncertainty, annotations).
This pulls the latest code (with cascade rules), clears only localization/repair/eval outputs,
and re-runs stages 4→5→6 in one cell. ~3-4 hours on A100-80GB for 500 questions.

In [36]:
import os, shutil

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab_musique.yaml'

# 1. Pull latest code with cascade-aware strategies
os.chdir(REPO_DIR)
!git pull

# 2. Clear ONLY stages 4-6 outputs (keep trajectories, uncertainty, annotations)
drive_base = '/content/drive/MyDrive/agent-repair-musique'
for base in [REPO_DIR, drive_base]:
    for d in ['outputs/localization', 'outputs/repairs', 'outputs/tables', 'outputs/figures']:
        p = os.path.join(base, d)
        if os.path.exists(p):
            shutil.rmtree(p)
            print(f'Cleared {p}')

# 3. Clear checkpoint files for stages 4-6 (these live in outputs/logs/)
for base in [REPO_DIR, drive_base]:
    for ckpt in ['outputs/logs/stage4_localize.jsonl',
                 'outputs/logs/stage5_repair.jsonl',
                 'outputs/logs/stage6_eval.jsonl']:
        p = os.path.join(base, ckpt)
        if os.path.exists(p):
            os.remove(p)
            print(f'Removed checkpoint: {p}')

print('\nStages 1-3 data preserved. Ready to re-run 4→5→6.')

Already up to date.
Cleared /content/drive/MyDrive/agent-repair-musique/outputs/localization
Cleared /content/drive/MyDrive/agent-repair-musique/outputs/repairs
Cleared /content/drive/MyDrive/agent-repair-musique/outputs/tables
Cleared /content/drive/MyDrive/agent-repair-musique/outputs/figures
Removed checkpoint: /content/drive/MyDrive/agent-repair-musique/outputs/logs/stage5_repair.jsonl

Stages 1-3 data preserved. Ready to re-run 4→5→6.


In [37]:
# Run stages 4 → 5 → 6 sequentially (run this and go to sleep)
import time

for stage in ['run_localize', 'run_repair', 'run_eval']:
    t0 = time.time()
    print(f'\n{"="*60}\n  {stage}\n{"="*60}')
    !python scripts/{stage}.py --config {CONFIG}
    elapsed = (time.time() - t0) / 60
    print(f'  ✓ {stage} done in {elapsed:.1f} min\n')

print('\n' + '='*60)
print('  ALL DONE — check results below')
print('='*60)


  run_localize
04:36:35 | INFO    | stage4 | config=config/config_colab_musique.yaml  base=/content/drive/MyDrive/agent-repair-musique
04:36:37 | INFO    | stage4 | records: 4250 (425 trajectories x 10 metrics)
                        argmax_top1  argmax_within1    mrr  threshold_hit  threshold_within1  top2_hit  top3_hit    n
metric                                                                                                               
self_consistency              0.129           0.322  0.328          0.169              0.454     0.231     0.346  425
max_token_prob_max            0.099           0.282  0.296          0.141              0.440     0.200     0.282  425
token_entropy_max             0.099           0.242  0.292          0.155              0.369     0.200     0.287  425
verbalized_confidence         0.085           0.242  0.323          0.122              0.339     0.221     0.379  425
token_entropy_mean_k50        0.073           0.268  0.268          0.089       

In [38]:
# View cascade-aware results
from src.utils import load_config
cfg = load_config(CONFIG)

summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())

import pandas as pd
results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)

import glob
from IPython.display import display, Image
figs = sorted(glob.glob(os.path.join(cfg.path('figures'), '*.png')))
for f in figs:
    print(f'\n--- {os.path.basename(f)} ---')
    display(Image(filename=f, width=800))

,strategy,fixed_%,95%_CI,avg_tokens,avg_tool_calls,hit_true_step_%
0,random_step,3.8,2.8-4.9%,216,3.90,14.8
1,full_restart,5.9,4.6-7.2%,348,6.62,42.8
2,oracle_targeted__bt2,5.6,4.4-6.8%,347,6.39,42.8
3,oracle_targeted__bt2__informed,5.6,4.4-7.0%,347,6.43,42.8
4,oracle_targeted__informed,4.9,3.8-6.0%,321,5.87,100.0
...,...,...,...,...,...,...
122,unc__verbalized_confidence__topk__bt2,3.8,2.8-4.9%,326,5.97,33.9
123,unc__verbalized_confidence__topk__bt2__informed,4.6,3.5-5.7%,323,6.02,33.9
124,unc__verbalized_confidence__topk__informed,3.9,2.9-5.0%,285,5.28,22.1
125,uncertainty_ensemble_any,16.2,14.3-18.3%,15317,281.18,84.9
